### Q3: What is the most informed action / series of actions in the dataset? For the person that made such an action, what was an "economic cost or risk" they exposed themselves to, beyond simply losing money on the trade?

Two deliverables: a specific action -- timestamp, markets, size, direction -- and then what
it cost that person to put it on, other than the money at risk.

#### What the data allows

`aggressor_buy_flag` gives which side initiated: True means the initiator lifted the offer
(bought YES), False means they hit the bid (sold YES). Verified against the book -- when
True the price is the best ask 94.9% of the time, when False the best bid 95.1%.

`player_id` and `player_name` are **100% null**. There is no trader identity anywhere in
the file, so we cannot follow an account across trades. That kills the obvious approach and
forces everything below: with no identity, "one person's action" has to be inferred from
timing and direction alone.

Every market is an "Over", so buying YES always means "more runs" and direction is
comparable across chains with no sign flipping.

#### The idea: score the action, not the trade

Someone with a view on scoring does not express it in one market. They hit several strikes
and several chains at once, and the exchange records that as many separate trades. Ranking
raw trades would miss it -- six 500-lot prints across six strikes look ordinary one at a
time and are one 3,000-lot action in reality.

So the unit of analysis is a **basket**: trades in the same direction inside a short window,
treated as one action.

**The family is {RFI, F5TOTAL, TOTAL}** -- 19 markets, 2,084 trades, 466,154 contracts. All
three measure runs over nested windows (1st inning, 5 innings, whole game), so a view about
scoring can be expressed in any of them.

The two team-total chains are dropped. PIT traded 1,270 contracts in six hours and MIL
14,681 -- too thin for anyone to get size on, and too few observations to say what an
outlier there would even look like. They also do not belong in the family: MIL runs and PIT
runs are different quantities, and being bullish on one is not being bullish on the other.

#### Three settings, and what each decides

**Grouping window = 1 second** -- how close in time trades must be to count as one action.
This encodes an assumption about the trader's execution technology that cannot be verified,
because there is no identity in the data: 1 second catches anyone trading through an API,
but a human clicking through a screen might take minutes to do the same thing. So it is a
parameter, and the analysis is re-run at 5s, 30s and 5min to check the answer does not
depend on it.

**Outlier fence = Tukey, `Q3 + 1.5 x IQR`**, computed on prior baskets only, so the score
is honest as a real-time statistic -- which Q4a will need. Mean and standard deviation are
unusable here because the distribution is extremely fat-tailed: the very baskets we are
hunting would inflate the threshold and hide the next one. Buys and sells are pooled for
the reference distribution, since there are only 307 buy-side baskets in the whole session
-- too few to place a quartile -- and the two sides are similar in shape at every quantile.

**Payoff window = 5 minutes**, fixed before looking at any results. Markout is
`(mid_after - mid_at_action) x (+1 if the initiator bought else -1)`: how far the market
moved their way afterwards.

Note what that last one implies. An action can only be *confirmed* informed after the fact,
which is the nature of the problem and exactly the gap Q4a has to close.


### Running it

Functions are in `utils.py` under the `Q3` banner. The pipeline is
`family_trades` -> `build_baskets` -> `score_baskets` -> `add_payoff`, wrapped by
`most_informed_actions`.

In [2]:
import numpy as np
import pandas as pd

from utils import (family_trades,               # slice to {RFI, F5TOTAL, TOTAL}
                   build_baskets,               # group trades by grouping window + direction
                   score_baskets,               # Tukey fence vs running distribution
                   add_payoff,                  # markout over the payoff window
                   most_informed_actions,       # end to end, ranked
                   grouping_window_robustness)  # does the winner survive other windows?

pd.set_option("display.width", 250)
pd.set_option("display.float_format", "{:,.2f}".format)

df_trades = pd.read_parquet("data/trades_823753_pregame.parquet")
df_books  = pd.read_parquet("data/orderbook_data_823753_pregame.parquet")

fam     = family_trades(df_trades)
baskets = build_baskets(fam)

print(f"family trades          : {len(fam):,} of {len(df_trades):,} "
      f"({100*len(fam)/len(df_trades):.0f}% -- team totals excluded)")
print(f"baskets at 1s          : {len(baskets):,} "
      f"(buy {(baskets.direction=='buy').sum()}, sell {(baskets.direction=='sell').sum()})")

family trades          : 2,084 of 2,232 (93% -- team totals excluded)
baskets at 1s          : 1,232 (buy 307, sell 925)


#### Ranked candidates

`x_median` is the basket's size divided by the running median of all prior baskets, so it
reads as "this action was N times the size of a typical one". Markouts are in cents,
positive meaning the market moved the initiator's way.

In [3]:
top, all_baskets = most_informed_actions(df_trades, df_books, top=10)

print(f"scoreable baskets      : {all_baskets.scoreable.sum():,}")
print(f"passing the Tukey fence: {(all_baskets.scoreable & all_baskets.is_outlier).sum()}")

cols = ["ts", "direction", "qty", "n_trades", "n_markets", "n_chains",
        "main_market", "x_median", "markout_5min", "markout_1min", "markout_30min"]
top[cols]

scoreable baskets      : 1,182
passing the Tukey fence: 230


,ts,direction,qty,n_trades,n_markets,n_chains,main_market,x_median,markout_5min,markout_1min,markout_30min
1171,2026-08-05 23:32:07+00:00,sell,"61,240.76",37,3,3,KXMLBRFI,"1,511.00",NaN,-0.00,NaN
433,2026-08-05 21:28:26+00:00,buy,"41,997.25",18,1,1,KXMLBTOTAL-8,"1,295.41",1.00,1.00,1.00
161,2026-08-05 19:05:34+00:00,sell,"27,430.25",1,1,1,KXMLBRFI,846.09,-0.00,-0.00,-0.00
676,2026-08-05 22:18:32+00:00,sell,"14,475.08",6,1,1,KXMLBRFI,451.08,-0.00,-0.00,-0.00
597,2026-08-05 22:05:17+00:00,buy,"13,833.00",5,1,1,KXMLBTOTAL-7,432.28,0.50,0.50,0.50
596,2026-08-05 22:05:06+00:00,buy,"11,365.84",9,1,1,KXMLBTOTAL-7,355.18,1.00,1.00,1.00
876,2026-08-05 22:59:23+00:00,buy,"10,485.20",7,1,1,KXMLBTOTAL-8,323.42,0.00,0.00,0.00
223,2026-08-05 19:45:01+00:00,sell,"7,295.71",1,1,1,KXMLBRFI,225.04,-0.00,-0.00,-0.00
1118,2026-08-05 23:29:56+00:00,sell,"8,363.61",35,4,3,KXMLBRFI,223.06,1.00,1.00,NaN
1031,2026-08-05 23:21:39+00:00,sell,"5,640.92",20,2,2,KXMLBF5TOTAL-4,169.24,-0.00,-0.00,NaN


#### The multi-chain subset

The question asks for an "action / series of actions". A basket touching several chains at
once is the strongest form of that: one view expressed simultaneously in markets that are
different contracts on the same underlying quantity.

In [5]:
o = all_baskets[all_baskets.scoreable & all_baskets.is_outlier]
print(f"outlier baskets      : {len(o)}")
print(f"  touching >1 market : {(o.n_markets > 1).sum()}")
print(f"  touching >1 chain  : {(o.n_chains > 1).sum()}")

o[o.n_chains > 1].sort_values("x_median", ascending=False).head(10)[cols]

outlier baskets      : 230
  touching >1 market : 53
  touching >1 chain  : 51


,ts,direction,qty,n_trades,n_markets,n_chains,main_market,x_median,markout_5min,markout_1min,markout_30min
1171,2026-08-05 23:32:07+00:00,sell,"61,240.76",37,3,3,KXMLBRFI,"1,511.00",NaN,-0.00,NaN
1118,2026-08-05 23:29:56+00:00,sell,"8,363.61",35,4,3,KXMLBRFI,223.06,1.00,1.00,NaN
1031,2026-08-05 23:21:39+00:00,sell,"5,640.92",20,2,2,KXMLBF5TOTAL-4,169.24,-0.00,-0.00,NaN
1096,2026-08-05 23:28:46+00:00,sell,"5,956.97",34,2,2,KXMLBRFI,165.47,1.00,-0.00,NaN
1218,2026-08-05 23:34:21+00:00,sell,"6,592.06",47,3,2,KXMLBRFI,162.65,NaN,-0.00,NaN
1194,2026-08-05 23:33:12+00:00,sell,"6,302.65",41,2,2,KXMLBRFI,155.51,NaN,-0.00,NaN
1145,2026-08-05 23:30:58+00:00,sell,"5,542.73",13,2,2,KXMLBRFI,139.23,NaN,-0.00,NaN
1119,2026-08-05 23:29:56+00:00,buy,"4,974.48",5,3,3,KXMLBTOTAL-8,131.91,0.00,0.00,NaN
1063,2026-08-05 23:25:04+00:00,sell,"4,373.08",2,2,2,KXMLBTOTAL-8,123.57,-0.00,-0.00,NaN
885,2026-08-05 23:01:06+00:00,sell,"2,111.10",3,2,2,KXMLBRFI,65.12,-0.00,-0.00,1.00


#### The final fifteen minutes

The ranked table above is dominated by a single stretch of the session, so it is worth
looking at that stretch directly rather than one basket at a time.

In [6]:
cut = pd.Timestamp("2026-08-05 23:20", tz="UTC")
late = fam[fam.ts >= cut]
sold   = late.loc[~late.aggressor_buy_flag, "qty"].sum()
bought = late.loc[ late.aggressor_buy_flag, "qty"].sum()

print(f"window                    : 23:20 -> {fam.ts.max():%H:%M:%S} UTC "
      f"({(fam.ts.max()-cut).total_seconds()/60:.0f} minutes)")
print(f"family trades in window   : {len(late)} of {len(fam)} "
      f"({100*len(late)/len(fam):.0f}% of the session)")
print(f"contracts sold vs bought  : {sold:,.0f} vs {bought:,.0f}  "
      f"({100*sold/(sold+bought):.0f}% sell)")
print(f"session-wide sell share   : "
      f"{100*fam.loc[~fam.aggressor_buy_flag,'qty'].sum()/fam.qty.sum():.0f}%")

mc = o[o.n_chains > 1]
print(f"multi-chain outliers here : {(mc.ts >= cut).sum()} of {len(mc)}, "
      f"of which {((mc.ts >= cut) & (mc.direction == 'sell')).sum()} are sells")

window                    : 23:20 -> 23:34:40 UTC (15 minutes)
family trades in window   : 605 of 2084 (29% of the session)
contracts sold vs bought  : 185,907 vs 12,031  (94% sell)
session-wide sell share   : 73%
multi-chain outliers here : 22 of 51, of which 17 are sells


### Answer: a sustained multi-chain selling campaign in the final fifteen minutes, peaking at 23:32:07 UTC

#### The single largest action

At **23:32:07 UTC**, eight minutes before first pitch, the tape records **37 sell prints
inside one second** totalling **61,241 contracts**, across **three different chains**: RFI,
F5TOTAL-4 and TOTAL-8.

Those are three contracts on the same question -- a run in the 1st inning, 4+ runs through
5 innings, 8+ runs in the game. Selling all three is one view, *this game will score less
than the market thinks*, expressed in three places at once.

It is **1,511x the running median basket**, 46% larger than the next largest action in the
dataset, and seven times the next multi-chain basket. Of the 230 baskets clearing the
outlier fence, only 51 touch more than one chain at all.

#### It is not an artifact of the grouping window

| grouping window | winner | direction | qty | chains |
| --- | --- | --- | --- | --- |
| 1s | 23:32:07 | sell | 61,241 | 3 |
| 5s | 23:32:05 | sell | 61,441 | 3 |
| 30s | 23:32:00 | sell | 69,702 | 3 |
| 5min | 23:30:00 | sell | 122,899 | 3 |

Same event at every setting, always selling, always three chains.

#### The better answer is the series, not the single print

The ranked table is dominated by one stretch of the session. From 23:20 to the end of the
data: **605 of 2,084 trades, 29% of the session inside 4% of the time**; **185,907 contracts
sold against 12,031 bought, 94% sell**, where the session-wide sell share is 73%; and **22
of the 51 multi-chain outlier baskets**, 17 of them sells.

So this is not one trader clipping once. It is fifteen minutes of concentrated,
overwhelmingly one-directional selling across all three chains, accelerating into first
pitch, with the 61,241 basket as its largest single moment.

One caveat on that window: 23:20 was chosen by eye after seeing where the cluster sat, and
it ends at 23:34:41 only because the data does. The concentration is real either way, but
the boundary is not a derived quantity.

#### What we cannot show: that it made money

Not established, and it would be wrong to claim it. The book feed stops at 23:34:41, so a
5-minute markout on this action needs a quote that does not exist. More fundamentally,
markout barely discriminates anywhere in this dataset: of the 199 outlier baskets with a
measurable 5-minute markout, **165 are exactly zero**, and the largest move in either
direction is one cent. Across the campaign RFI drifted 0.405 to 0.395 while F5TOTAL-4 and
TOTAL-8 did not move at all.

So "informed" here means the action has the *shape* of someone acting on a view -- extreme
size, coherent direction across three correlated chains, timed into the last minutes before
first pitch -- not that we proved they were right. A large uninformed bettor cannot be ruled
out. The contracts settle on the actual run total and the data ends before the first pitch
is thrown, so the one thing that would truly confirm it is outside the dataset entirely.

---

### The economic cost and risk taken on, beyond losing money

The money at risk is the boring part and the question excludes it: 61,241 contracts sold at
a volume-weighted `$0.427` collects `$26,138` in premium against a maximum liability of
`$61,241`, so the position risks `$35,103`. What follows is what the trader gave up *by the
act of trading itself*.

**1. They told the market what they knew.** The 37 prints are public tape. Anyone watching
sees a large seller working three correlated chains minutes before first pitch. The only way
to convert private information into money is to reveal you have it, and the edge is worth
less the moment it is exercised.

**2. The position cannot be exited.** 61,241 contracts is a large slice of the 466,154 the
whole family traded all session. Getting out means buying back into a book they just
emptied, at prices they themselves moved. Practically it is a hold to settlement, which
turns a probabilistic view into an all-or-nothing outcome: a view that is 70% likely to be
right becomes a coin flip they cannot unwind or hedge.

**3. The maker learns they were adversely selected, and widens.** Q1 found this chain quoted
with zero arbitrage violations and perfect internal consistency, including in strikes that
never traded once -- the signature of one maker pricing the whole ladder off a fitted
distribution. A take this size tells that maker their quotes were stale relative to
somebody's information, and the rational response is to widen, pull size, or skew away. The
edge is burned on the first clip.

**4. The trade telegraphs across all nineteen markets, not just the three they hit.** This
is the specific cost of going multi-chain and the one most people would miss. Because the
ladder is internally consistent, the maker cannot reprice RFI, F5TOTAL-4 and TOTAL-8 in
isolation -- moving any strike forces a coherent move in every correlated strike, or the
ladder breaks its own monotonicity and nesting constraints. So the trader reveals not just
"someone sold RFI" but the shape of their whole opinion about scoring in this game, across
every strike on the board, instantly and for free.

**5. Legal and regulatory exposure.** If the edge came from material non-public information
-- a late scratch, a lineup change, something from inside the club -- the downside is
disgorgement, fines, exchange suspension or prosecution rather than a losing trade. Timing
eight minutes before first pitch, exactly when late team news lands, is the pattern that
attracts that scrutiny. The CFTC's Enforcement Division issued a Prediction Markets Advisory
in February 2026 describing MNPI-based event-contract trading as pursuable under CEA section
6(c)(1) and Rule 180.1, on a misappropriation theory rather than classic securities insider
trading. Three cases so far: a YouTube channel editor who traded ahead of video releases
(Kalshi, 2025, a `$20,397.58` penalty and a two-year suspension); a political candidate
trading his own race (Kalshi, May 2025); and a Google engineer charged by the CFTC and DOJ
over Polymarket contracts tied to internal "Year in Search" data (May 2026), the first case
involving a private-sector employee trading on internal work knowledge.

**6. Being wrong in public, at size, in front of a maker who remembers.** In a venue this
concentrated, a trader who sprays size and turns out to be wrong finds their flow priced
accordingly, and the cost shows up as worse fills on every later trade rather than as one
visible loss.
